In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langgraph.graph import StateGraph,START,END
from pydantic import BaseModel
from langchain_groq import ChatGroq
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from typing import Annotated

In [3]:
class chatState(BaseModel):
    messages: Annotated[list, add_messages]

In [4]:
llm = ChatGroq(
    model="openai/gpt-oss-120b"
)

In [10]:
def ChatBotNode(state:chatState):
    res = llm.invoke(state.messages)
    return {
        "messages": [res]
    }

In [11]:
memory = InMemorySaver()

In [12]:
graph = StateGraph(chatState)
graph.add_node("chatBot", ChatBotNode)

graph.add_edge(START, "chatBot")
graph.add_edge("chatBot", END)

graph = graph.compile(checkpointer=memory)

In [15]:
res = graph.invoke(
    {"messages":[{"role":"user", "content":"Hey im Pranav"}]},
    {"configurable":{"thread_id":"my-bot-1"}}
    )

In [16]:
res["messages"][-1].content

'Hey Pranav! 👋 Nice to meet you. How’s your day going? Anything you’d like to chat about or need help with?'